# Разбор логов автопродлений подписки

Чистый Python, без pandas и без регулярных выражений в логике счёта. Данные: `data/auto_purchase.log`, 47 строк за неделю 3-9 октября 2022.

Постановка в `data/02_обработка_логов.ТЗ.md`.

## Формат строки

```
INFO | 2022-10-03 00:04:12,118 | file: demon_auto_purchase_bundle.py | line: 86 | [demon] Обновляем подписку пользователю id: 4412, payment_method_id: 9f2c1ab4
ERROR | 2022-10-03 00:07:55,440 | file: demon_auto_purchase_bundle.py | line: 93 | [demon] У пользователя с id: 3390 - ошибка при списании:
INFO | 2022-10-03 22:00:03,301 | file: demon_auto_purchase_bundle.py | line: 61 | [demon] Итог по подпискам за Cегодня 2022-10-03 количество клиентов с подпиской: 18
```

Три вида записей: попытка списания, ошибка списания, итог за день.

In [ ]:
LOG = "../data/auto_purchase.log"

with open(LOG, encoding="utf-8") as f:
    lines = f.readlines()

print("строк в логе:", len(lines))
print()
for line in lines[:5]:
    print(line.rstrip())

## Почему не срез по индексам

Первое, что приходит в голову, это `line[7:17]` для даты. Оно не работает.

Длина префикса разная: `INFO | ` это 7 символов, `ERROR | ` это 8. Сдвиг на один символ, и дата режется как ` 2022-10-0`, `strptime` падает. На первой же строке с ERROR.

Поэтому дата и признаки достаются поиском подстроки, а не срезом. Это медленнее на символах, но не ломается при смене уровня логирования, а в логах уровень меняется регулярно.

In [ ]:
def parse_date(line):
    """Дата в любой строке лога. Ищем по разделителю, а не по индексу:
    у INFO и ERROR разная длина префикса."""
    parts = line.split("|")
    return parts[1].strip().split()[0]

print("первые даты:", [parse_date(l) for l in lines[:6]])

# проверка, что срез по индексу действительно ломается
try:
    [l[7:17] for l in lines if l.startswith("ERROR")][0]
    print("срез отработал, но дата подозрительная:", [l[7:17] for l in lines if l.startswith("ERROR")][0])
except Exception as e:
    print("срез ломается:", type(e).__name__)

## Задача 1. Успешные и неуспешные списания

Самое тонкое место здесь. В логе `INFO` пишется **каждая** попытка списания, включая те, что закончились ошибкой. Об этом прямо сказано в описании задачи, но по виду лога это не очевидно: строка выглядит как обычное успешное обновление.

Значит считать надо так: из числа `INFO`-попыток вычесть число `ERROR`-записей. Если считать их раздельно, доля успешных получится завышенной, а в пределе любой отчёт об оттоке подписок будет врать в сторону «всё хорошо».

In [ ]:
ATTEMPT = "Обновляем подписку"
FAILURE = "ошибка при списании:"

def count_success_and_failure(path: str) -> tuple[int, int]:
    attempts = failures = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("ERROR") and FAILURE in line:
                failures += 1
            elif line.startswith("INFO") and ATTEMPT in line:
                attempts += 1
    return attempts - failures, failures

success, failure = count_success_and_failure(LOG)
print(f"успешных списаний: {success}")
print(f"ошибочных списаний: {failure}")
print(f"доля успешных: {success/(success+failure):.4f}")

32 попытки, 8 ошибок, доля успешных 0.75.

Разделение на эти две цифры отвечает на разные вопросы. Много ошибок значит проблему в биллинге или в платёжных методах, вопрос к разработке. Мало ошибок при низком числе успешных значит, что не покупают, вопрос к продукту и маркетингу. По одной агрегированной цифре это не различить.

## Задача 2. Динамика подписок

В логе есть итоговая строка за каждый день, в ней число клиентов с подпиской. По ней считаются среднее и медиана на накопленном ряду.

Здесь есть ловушка в словаре: если по одному дню встречается несколько итоговых строк, последняя просто затрёт предыдущую. В исходной версии решений брался максимум из встреченных значений, потому что в логах бывают дубли перезаписи.

In [ ]:
import re

DAY = re.compile(r"Cегодня (\d{4}-\d{2}-\d{2}).*подпиской: (\d+)")

def daily_subscriptions(path: str) -> dict[str, int]:
    result: dict[str, int] = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = DAY.search(line)
            if m:
                date, count = m.group(1), int(m.group(2))
                # дубль итога за один день: берём максимум, не последнее значение
                result[date] = max(result.get(date, 0), count)
    return {d: result[d] for d in sorted(result)}

subs = daily_subscriptions(LOG)
for date, value in subs.items():
    print(f"{date}  {value:>3} подписчиков")

In [ ]:
def mean(values):
    return round(sum(values) / len(values), 2)

def median(values):
    ordered = sorted(values)
    n = len(ordered)
    if n % 2:
        return ordered[n // 2]
    # чётная длина: среднее двух центральных
    return int((ordered[n // 2 - 1] + ordered[n // 2]) / 2)

values = list(subs.values())
rolling_mean = [mean(values[:i]) for i in range(1, len(values) + 1)]
rolling_median = [median(values[:i]) for i in range(1, len(values) + 1)]

print("накопленное среднее:  ", rolling_mean)
print("накопленная медиана:  ", rolling_median)

Среднее и медиана почти не расходятся: 22.14 против 22 на седьмой день. На коротком ряду без выбросов так и должно быть.

Разница проявилась бы на данных с выбросами, и вот в этом различие существенное. Если в какой-то день подписок 500 при обычных 20, среднее подскочит, а медиана останется на месте. Именно поэтому для «сколько у нас подписчиков» медиана надёжнее: на неё не влияет единичный всплеск.

## Задача 3. Дни недели

Считается число успешных списаний по дням недели.

In [ ]:
from datetime import datetime

WD = ["Понедельник", "Вторник", "Среда", "Четверг", "Пятница", "Суббота", "Воскресенье"]

def renewals_by_weekday(path: str) -> dict[str, int]:
    counter = {d: 0 for d in WD}
    with open(path, encoding="utf-8") as f:
        for line in f:
            date = parse_date(line)
            weekday = datetime.strptime(date, "%Y-%m-%d").weekday()
            if line.startswith("ERROR") and FAILURE in line:
                counter[WD[weekday]] -= 1
            elif line.startswith("INFO") and ATTEMPT in line:
                counter[WD[weekday]] += 1
    return counter

by_wd = renewals_by_weekday(LOG)
for day, value in by_wd.items():
    bar = "#" * max(value, 0)
    print(f"{day:<12} {value:>3}  {bar}")

В логе неделя целиком, поэтому по каждому дню недели ровно одно наблюдение и распределение по дню недели плюс шум от того, что именно выпало на этот день. Читать из этого «пятница лучшая» нельзя.

Для нормального вывода нужно несколько недель, и тогда сравнивать надо среднее по дню недели, а не сумму.

In [ ]:
# так честнее: среднее за день недели, а не сумма
dates = sorted({parse_date(l) for l in lines if l.startswith(("INFO", "ERROR"))})
print("дней в данных:", len(dates), dates[0], "по", dates[-1])
print()
print("Один день недели = одно наблюдение, поэтому распределение выше")
print("содержит шум от того, какая неделя попала в выборку.")

## Итоги

| Метрика | Значение |
|---|---|
| Строк в логе | 47 |
| Попыток списания | 32 |
| Успешных | 24 |
| Ошибочных | 8 |
| Доля успешных | 0.75 |
| Подписчиков на 9 октября | 23 |
| Накопленное среднее | 22.14 |
| Накопленная медиана | 22 |

Что стоит вынести:

1. `INFO` в логе пишется на каждую попытку, включая неуспешные. Считать успех и ошибку раздельно нельзя, доля успеха всегда завышена. В исходной версии это давало бы 1.0 вместо 0.75.
2. Уровень логирования меняется, поэтому дата не берётся срезом по индексу. `ERROR | ` длиннее `INFO | ` на символ, и `line[7:17]` на ERROR-строках падает.
3. Среднее и медиана накапливаются по накопленному ряду. Медиана нужна ради устойчивости к выбросам, на коротком ряду без них разницы не видно, но на реальном журнале она появляется сразу.
4. Дни недели по одной неделе читать нельзя: одно наблюдение на день, вся разница это шум выборки.